<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 31 · Tool-Using Agents for Fixed-Income Analysis

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch31_tool_using_agents_fixed_income_analysis.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

After working through this notebook, you will be able to:

- draw the boundary between planning language and numerical tool output in a fixed-income agent workflow;
- describe a tool registry with task name, input schema, output units, and failure modes, and separate read-only tools from state-changing actions under least privilege;
- read an agent validation log and reconcile each drafted claim against the deterministic tool value;
- explain why a failed reconciliation must block or reroute the task before any sentence reaches a decision maker; and
- identify the controls that stop tool-chain errors from propagating from curve lookup to pricing to stress to report.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script reads the frozen validation log in `data/ch31_agent_validation_log.csv`, which stores four agent claims about a fixed-income analysis: the 10-year rate in percent, a clean bond price, a spread-shock loss in millions, and the number of failed checks. The script isolates every claim whose drafted value differs from the tool value and counts the pass and fail outcomes. It makes no network calls and invents no numbers: the log is the controlled record. Read the printed table alongside the chapter's reconciliation logic; the two failing rows are the lesson.

In [ ]:
SCRIPT = ROOT / "code" / "ch31_tool_using_agents_fixed_income_analysis.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Check the output before you trust it:

- Confirm the units of each claim: the 10-year rate is quoted in percent, the clean price in currency units per 100 of face value, the stress loss in millions with the sign convention that negative means loss.
- Verify that the failing stress claim, -2.35 from the tool against -2.30 in the draft, is a real mismatch rather than a documented rounding rule, and that the report-summary claim, one failed check in the log against zero in the draft, contradicts the log itself.
- Note that a single wrong curve node can propagate through pricing, stress, and reporting, so reconciliation belongs at each tool boundary, not only at the final report.
- State which curve source, date, and measure each tool value refers to before using it in prose.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch31_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Replay the claim reconciliation and verify that both the stress claim and report-summary claim are routed to review.

In [ ]:
log = pd.read_csv(DATA / "ch31_agent_validation_log.csv")
failed = log.loc[log.status.eq("fail"),
                 ["step", "claim", "tool_value", "draft_value"]]
summary = log.groupby("status").size().rename("checks").reset_index()
failed, summary

## Mini-lab: An agent tool-call loop on a mock calculator

This lab isolates the loop at the heart of the chapter: an agent follows a frozen plan, requests calculations from a tool registry, and every drafted number is reconciled against the tool output. The mock registry has two deterministic tools: a curve lookup returning a 10-year zero rate of 4.18 percent and a pricer that discounts 100 of face value at that rate with annual compounding over 10 years. The agent's second claim, drafted from language alone, is checked against the pricer result.

In [ ]:
import pandas as pd

def curve_lookup():  # 10y zero rate in decimals
    return 0.0418

def clean_price(rate):  # mock pricer: annual compounding
    return round(100 / (1 + rate) ** 10, 2)

plan = [("curve", None), ("valuation", "curve")]  # agent plan
state, rows = {}, []
for action, arg in plan:  # tool-call loop
    tool = curve_lookup() if arg is None else clean_price(state[arg])
    state[action] = tool
    rows.append((action, tool))
draft = 70.0  # drafted, not computed
log = pd.DataFrame(rows, columns=["step", "tool_value"])
log["draft_value"] = [state["curve"], draft]
mismatch = log.tool_value != log.draft_value
log["status"] = ["fail" if m else "pass" for m in mismatch]
log

### What to observe

The curve tool returns 0.0418 and the pricer returns 66.4, so the curve claim passes. The valuation claim drafted at 70.0 fails reconciliation even though both tools ran correctly, and the workflow must route that item to review before any sentence uses it.

## Exercises

1. **Reconcile every claim.** Extend the loop so each step carries a drafted claim, and print only the rows where the drafted value differs from the tool value. *Hint:* Compare the two columns with a boolean mask on the log DataFrame.
2. **Add a stress tool.** Add a mock spread-shock tool that maps a duration and a 50 basis point shock to a loss in currency units, and route it through the same loop. *Hint:* Keep the sign convention explicit: negative values denote losses.
3. **Separate read-only from state-changing tools.** Split the registry into lookups that only read and actions that would change state, and refuse the second category without approval. *Hint:* Model approval as a separate boolean gate that the loop must check.
4. **Audit the loop.** Record the task name, input, tool output, unit, and status for each call, and explain which fields make a failed check reproducible. *Hint:* Add columns to the log rather than printing free text.
5. **Break the chain deliberately.** Corrupt the curve input and trace how the error propagates into the price and any downstream claim. *Hint:* Compare the validation results before and after the corruption.

For more detail, see Chapter 31 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">